# MaizeGuard: maize leaf disease classification (ML pipeline summative)

**Problem.** My Introduction to ML summative predicted which Rwandan farmers use improved seed. Improved, disease-tolerant seed only pays off if farmers can recognise disease in the field early. This notebook builds an **image classifier** that tells a healthy maize leaf from three common diseases (Blight, Common Rust, Gray Leaf Spot), so the pipeline can advise farmers and agro-dealers.

**Contents**
1. Data acquisition
2. Data exploration and feature interpretation (3 features)
3. Data preprocessing
4. Model creation (transfer learning + regularisation)
5. Training with early stopping, then fine-tuning
6. Model testing: accuracy, loss, precision, recall, F1, confusion matrix
7. Prediction function
8. Retraining function (uses the saved model as the pre-trained model)

> Run on Google Colab with a GPU: *Runtime > Change runtime type > T4 GPU*.

## 0. Setup
Clone the repository so the notebook uses the same `src/` code as the API. **Replace the URL with your own repo.**

In [ ]:
import os
REPO_URL = "https://github.com/<your-username>/MaizeGuard.git"   # <-- change me
if not os.path.exists("src"):                 # already inside the repo? then skip
    if not os.path.exists("MaizeGuard"):
        !git clone {REPO_URL}
    %cd MaizeGuard
!pip -q install kagglehub

In [ ]:
import json, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from PIL import Image

from src.preprocessing import (CLASS_NAMES, IMG_SIZE, TRAIN_DIR, TEST_DIR, list_images,
                               make_dataset, split_train_val, decode_image, augment)
from src.model import (build_model, compile_model, unfreeze_top_layers, get_callbacks,
                       evaluate_model, save_metrics, retrain_model, MODEL_PATH)
from src.prediction import predict_image
from src.insights import compute_insights

tf.random.set_seed(42); np.random.seed(42); random.seed(42)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Data acquisition
Dataset: **Corn or Maize Leaf Disease Dataset** (Kaggle, `smaranjitghose/corn-or-maize-leaf-disease-dataset`), about 4,000 leaf photos in four folders, built from PlantVillage and PlantDoc. `kagglehub` downloads it without an API key.

`scripts/prepare_data.py` then finds the four class folders, **drops corrupt files** and makes a fixed **80 / 20 train / test split** into `data/train` and `data/test`.

In [ ]:
import kagglehub
raw_dir = kagglehub.dataset_download("smaranjitghose/corn-or-maize-leaf-disease-dataset")
print("Downloaded to:", raw_dir)
if not list_images(TRAIN_DIR)[0]:             # only split once
    !python scripts/prepare_data.py "{raw_dir}"

In [ ]:
train_paths, train_labels = list_images(TRAIN_DIR)
test_paths,  test_labels  = list_images(TEST_DIR)
print(f"train images: {len(train_paths)}   test images: {len(test_paths)}")

## 2. Data exploration
### 2.1 What do the classes look like?

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(13, 10))
for r, name in enumerate(CLASS_NAMES):
    files = [p for p, l in zip(train_paths, train_labels) if l == r]
    for c, p in enumerate(random.sample(files, 5)):
        axes[r, c].imshow(Image.open(p)); axes[r, c].axis("off")
    axes[r, 0].set_title(name, loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

### 2.2 Feature 1: class balance

In [ ]:
insights = compute_insights(TRAIN_DIR)        # also saves models/data_insights.json for the dashboard
counts = pd.Series(insights["class_counts"])
ax = counts.plot.bar(color="seagreen", figsize=(6, 3.5), rot=0, title="Training images per class")
ax.bar_label(ax.containers[0]); plt.show()
print("imbalance ratio (largest / smallest):", round(counts.max() / counts.min(), 2))

**Interpretation (write in your own words after looking at the chart).** Which class is smallest? A class with fewer images is seen less often in training, so its recall is usually the weakest. That is why this notebook reports per-class precision / recall and macro F1, not accuracy alone.

### 2.3 Feature 2: colour (how green is the leaf?)

In [ ]:
feats = pd.DataFrame(insights["features"]).T
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
feats[["red", "green", "blue"]].plot.bar(ax=ax[0], color=["tab:red", "tab:green", "tab:blue"], rot=0,
                                         title="Mean colour channel value per class")
feats["green_ratio"].plot.bar(ax=ax[1], color="seagreen", rot=0, title="Share of green = G / (R+G+B)")
plt.show(); feats.round(3)

**Interpretation.** Disease replaces green chlorophyll with brown, grey or rust-coloured tissue. Check: does `Healthy` have the highest green share? Which disease is closest to it? Classes with close colour values are harder to separate on colour alone.

### 2.4 Feature 3: brightness and contrast (surface texture)

In [ ]:
feats[["brightness", "contrast"]].plot.bar(figsize=(7, 3.5), rot=0,
      title="Brightness (mean grey level) and contrast (std of grey level)")
plt.show()

**Interpretation.** Contrast measures how much light/dark variation there is inside one photo. Lesions and rust pustules create patches, so diseased leaves tend to be less uniform than healthy ones. Compare your numbers: which two classes are most alike on BOTH colour and contrast? Expect those to be the most confused pair in the confusion matrix later.

## 3. Data preprocessing
Every image goes through the same steps (code in `src/preprocessing.py`):

| Step | What | Why |
|---|---|---|
| 1 | Validate / drop corrupt files | a broken file would crash training |
| 2 | Decode to RGB (3 channels) | some files are PNG or greyscale |
| 3 | Resize to 224 x 224 | input size of MobileNetV2 |
| 4 | One-hot encode the label | needed for categorical cross-entropy, precision and recall |
| 5 | Augment (training only): flips, 90-degree rotations, brightness, contrast | more variety, less overfitting; field photos differ in angle and light |
| 6 | Scale pixels to [-1, 1] | done **inside the model** (`Rescaling` layer), so the API cannot forget it |
| 7 | Batch (32) and prefetch | speed |

The training folder is split again into **80% train / 20% validation** (stratified). The test folder is never touched until section 6.

In [ ]:
tr_p, tr_l, va_p, va_l = split_train_val(train_paths, train_labels, val_fraction=0.2)
train_ds = make_dataset(tr_p, tr_l, training=True)
val_ds   = make_dataset(va_p, va_l)
print(f"train={len(tr_p)}  validation={len(va_p)}  test={len(test_paths)}")
for x, y in train_ds.take(1):
    print("batch images:", x.shape, x.dtype, "| pixel range:", float(tf.reduce_min(x)), "-", float(tf.reduce_max(x)))
    print("batch labels:", y.shape, "(one-hot)")

In [ ]:
# What augmentation does to ONE image
img = decode_image(tf.io.read_file(tr_p[0]))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(img.numpy().astype("uint8")); axes[0].set_title("original (resized)")
for ax in axes[1:]:
    ax.imshow(augment(img).numpy().astype("uint8")); ax.set_title("augmented")
for ax in axes: ax.axis("off")
plt.show()

## 4. Model creation
**Transfer learning.** MobileNetV2 pre-trained on ImageNet is the feature extractor (frozen at first). It is small and fast, which matters for a cheap cloud server. On top sits a small head.

Optimisation techniques used:
* **Pre-trained model** (MobileNetV2)
* **Regularisation:** Dropout (0.3) twice + L2 weight penalty on the dense layer
* **Optimizer:** Adam, learning rate 1e-3, then 1e-5 for fine-tuning
* **Early stopping** on validation loss (patience 3, restores best weights)
* **ReduceLROnPlateau** learning-rate schedule
* **Data augmentation**

In [ ]:
model = build_model(learning_rate=1e-3, dropout=0.3, l2=1e-4)
model.summary()

## 5. Training
### 5.1 Stage 1: train only the new head (base frozen)

In [ ]:
history1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=get_callbacks())

### 5.2 Stage 2: fine-tune the top 30 layers of MobileNetV2 with a very small learning rate

In [ ]:
unfreeze_top_layers(model, n_layers=30)
compile_model(model, learning_rate=1e-5)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=get_callbacks())

In [ ]:
hist = {k: history1.history[k] + history2.history[k] for k in history1.history if k in history2.history}
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, key in zip(ax, ["loss", "accuracy"]):
    a.plot(hist[key], label="train"); a.plot(hist["val_" + key], label="validation")
    a.axvline(len(history1.history["loss"]) - 0.5, ls="--", c="grey", label="fine-tuning starts")
    a.set_title(key); a.set_xlabel("epoch"); a.legend()
plt.show()

**Read the curves.** If validation loss follows training loss, the model is not overfitting. A widening gap would mean overfitting (more dropout or augmentation needed).

## 6. Model testing
Evaluated on the **held-out test set** with 5 metrics: accuracy, loss (cross-entropy), precision, recall and F1 (macro average, so every class counts equally), plus a per-class report and the confusion matrix.

In [ ]:
metrics = evaluate_model(model, test_paths, test_labels)
pd.Series({k: metrics[k] for k in ["accuracy", "loss", "precision", "recall", "f1"]}).round(4).to_frame("test set")

In [ ]:
pd.DataFrame(metrics["per_class"]).T.round(3)

In [ ]:
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(metrics["confusion_matrix"], annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (test set)"); plt.show()

**Interpretation (fill in with your numbers).** Which class has the lowest recall? Which pair is confused most? Does this match the class-balance and colour/contrast findings from section 2? For a farmer, a *missed disease* (low recall on a disease class) costs more than a false alarm, so recall on the disease classes matters most.

### Save the model and its metrics
The `.h5` file is what the API loads. `metrics.json` is what the dashboard shows as the evaluation of the production model.

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True)
model.save(str(MODEL_PATH))
save_metrics(metrics, {k: [float(v) for v in vals] for k, vals in hist.items()})
print("saved:", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

## 7. Prediction function
`src/prediction.py` loads the saved file and classifies one image from raw bytes, exactly as the API does. One test image per class:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for idx, ax in enumerate(axes):
    p = random.choice([q for q, l in zip(test_paths, test_labels) if l == idx])
    res = predict_image(Path(p).read_bytes())
    ok = res["label"] == CLASS_NAMES[idx]
    ax.imshow(Image.open(p)); ax.axis("off")
    ax.set_title(f"true: {CLASS_NAMES[idx]}\npred: {res['label']} ({res['confidence']:.0%})",
                 color="green" if ok else "red")
plt.show()
res

## 8. Retraining function
`retrain_model()` is what the **Retrain** button in the dashboard calls. It:
1. loads the **saved model as the pre-trained model** (it does not start from ImageNet again),
2. preprocesses the newly uploaded images with the same pipeline,
3. mixes in a small sample of the original training data so old knowledge is not forgotten,
4. fine-tunes with a small learning rate and early stopping,
5. evaluates before and after on the test set and **only promotes the new model if accuracy did not drop**.

Here we simulate an upload with 10 validation images per class.

In [ ]:
new_p, new_l = [], []
for idx in range(len(CLASS_NAMES)):
    group = [p for p, l in zip(va_p, va_l) if l == idx][:10]
    new_p += group; new_l += [idx] * len(group)

result = retrain_model(new_p, new_l, epochs=3)
result

## Download the files for the repo
Download `models/maize_leaf_model.h5`, `models/metrics.json` and `models/data_insights.json` and commit them to your GitHub repo (plus the `data/train` and `data/test` folders).

In [ ]:
!zip -q -r artifacts.zip models/maize_leaf_model.h5 models/metrics.json models/data_insights.json
try:
    from google.colab import files; files.download("artifacts.zip")
except ImportError:
    print("Not on Colab: files are in models/")